# RAVE Training — In C / Mundo Real

Entrenamiento del modelo RAVE con el corpus de sesiones de incsynth.

**Antes de correr este notebook:**
1. Subir la carpeta `corpus/chunks/` a Google Drive en `Mi unidad/incsynth/corpus/chunks/`
2. Activar GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU
3. Correr las celdas en orden

In [ ]:
# Celda 1 — Verificar GPU
import torch
if torch.cuda.is_available():
    print(f'GPU OK: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No hay GPU disponible. Activar T4 en Entorno de ejecución.')

In [ ]:
# Celda 2 — Instalar RAVE
!pip install acids-rave --quiet
print('acids-rave instalado')

In [ ]:
# Celda 3 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_CORPUS = '/content/drive/MyDrive/incsynth/corpus/chunks'
assert os.path.isdir(DRIVE_CORPUS), f'No encontré {DRIVE_CORPUS} — verificar que subiste la carpeta'

In [ ]:
# Celda 4 — Copiar corpus a disco local (más rápido para training)
!cp -r {DRIVE_CORPUS} /content/corpus_chunks
wav_count = len([f for f in os.listdir('/content/corpus_chunks') if f.endswith('.wav')])
print(f'Corpus copiado: {wav_count} archivos WAV')

In [ ]:
# Celda 5 — Preprocesar corpus
!rave preprocess \
    --input_path /content/corpus_chunks/ \
    --output_path /content/corpus_prep/
print('Preprocesado listo')

In [ ]:
# Celda 6 — Entrenar RAVE
# max_steps=1000000 es suficiente para una primera versión funcional (~3-4h en T4)
# val_every=10000 guarda checkpoint cada 10k pasos
!rave train \
    --config v2 \
    --db_path /content/corpus_prep/ \
    --name incsynth_rave \
    --max_steps 1000000 \
    --val_every 10000
print('Training terminado')

In [ ]:
# Celda 7 — Encontrar el directorio del run
import glob
runs = glob.glob('/content/runs/incsynth_rave*')
assert runs, 'No se encontró el directorio del run'
RUN_PATH = sorted(runs)[-1]
print(f'Run: {RUN_PATH}')
!ls {RUN_PATH}/

In [ ]:
# Celda 8 — Exportar modelo (.ts streaming + .onnx)
!rave export --run {RUN_PATH} --streaming
!rave export_onnx --run {RUN_PATH}
!ls {RUN_PATH}/*.ts {RUN_PATH}/*.onnx 2>/dev/null || ls {RUN_PATH}/

In [ ]:
# Celda 9 — Guardar modelo en Google Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/incsynth/rave_model'
!mkdir -p {DRIVE_OUTPUT}
!cp {RUN_PATH}/*.ts {DRIVE_OUTPUT}/ 2>/dev/null
!cp {RUN_PATH}/*.onnx {DRIVE_OUTPUT}/ 2>/dev/null
!ls {DRIVE_OUTPUT}/
print('Modelo guardado en Drive. Podés descargarlo desde Mi unidad/incsynth/rave_model/')